# Imports and Config

In [1]:
import os, json, csv, gc, shutil
from dataclasses import dataclass

import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset
from transformers import AutoTokenizer, Trainer, TrainingArguments, DataCollatorWithPadding
from safetensors.torch import load_file

from coral_pytorch.losses import corn_loss
from coral_pytorch.dataset import corn_label_from_logits
from model import T5Gemma2OrdinalClassifier
from metrics import evaluate_predictions, LABEL_NAMES

LEVELS = ["A2", "A2+", "B1", "B1+", "B2", "B2+", "C1", "C1+"]
LABEL2ID = {lvl: i for i, lvl in enumerate(LEVELS)}
NUM_CLASSES = len(LEVELS)


def corn_probas(logits):
    """(N, K-1) CORN logits -> (N, K) class distribution, rows sum to 1."""
    cond = torch.sigmoid(logits)
    cum = torch.cumprod(cond, dim=1)
    p = torch.zeros(logits.size(0), logits.size(1) + 1)
    p[:, 0] = 1.0 - cum[:, 0]
    p[:, 1:-1] = cum[:, :-1] - cum[:, 1:]
    p[:, -1] = cum[:, -1]
    return p.clamp_min(0.0)


@dataclass
class CFG:
    mode: str = "corn"
    model_id: str = "google/t5gemma-2-4b-4b"
    eval_path: str = "data/val.jsonl"
    text_col: str = "text"
    label_col: str = "label"
    max_length: int = 1024
    batch_size: int = 8
    bf16: bool = True
    output_dir: str = "runs/t5gemma2_corn"

cfg = CFG()

# Data set and Tokenizer

In [2]:
tokenizer = AutoTokenizer.from_pretrained(cfg.model_id)


class JsonlDataset(Dataset):
    def __init__(self, path, tokenizer, cfg):
        self.rows = [json.loads(l) for l in open(path, encoding="utf-8")]
        self.tok, self.cfg = tokenizer, cfg

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, i):
        r = self.rows[i]
        enc = self.tok(r[self.cfg.text_col], truncation=True,
                       max_length=self.cfg.max_length)
        enc["labels"] = LABEL2ID[r[self.cfg.label_col]]
        return enc


eval_ds = JsonlDataset(cfg.eval_path, tokenizer, cfg)
print(len(eval_ds), "eval examples")

599 eval examples


# Trainer Class

In [3]:
class OrdinalTrainer(Trainer):
    def __init__(self, *a, mode="corn", **k):
        super().__init__(*a, **k)
        self.mode = mode

    def compute_loss(self, model, inputs, return_outputs=False, **kw):
        labels = inputs.pop("labels")
        out = model(**inputs)
        logits = out.logits
        loss = (corn_loss(logits, labels.long(), NUM_CLASSES) if self.mode == "corn"
                else torch.nn.functional.cross_entropy(logits, labels.long()))
        return (loss, out) if return_outputs else loss


def compute_metrics(eval_pred):
    logits = eval_pred.predictions
    if isinstance(logits, tuple):
        logits = logits[0]
    logits = torch.as_tensor(np.asarray(logits, dtype=np.float32))
    preds = (corn_label_from_logits(logits).numpy() if cfg.mode == "corn"
             else logits.argmax(-1).numpy())
    return evaluate_predictions(eval_pred.label_ids, preds)

# Create Backup *to preserve previous results

In [4]:
for run in ["runs/t5gemma2_corn", "runs/t5gemma2_nominal"]:
    for f in ["dev_metrics.json", "dev_predictions.csv"]:
        src = os.path.join(run, f)
        if os.path.exists(src) and not os.path.exists(src + ".bak"):
            shutil.copy(src, src + ".bak")
            print("backed up", src)

# Reusable Run Function and Run CORN

In [5]:
def run_condition(mode, output_dir, ckpt_name="checkpoint-1900"):
    cfg.mode, cfg.output_dir = mode, output_dir
    ckpt = os.path.join(output_dir, ckpt_name)
    print(f"=== {mode} | {ckpt} ===")

    dtype = torch.bfloat16 if cfg.bf16 else torch.float32
    model = T5Gemma2OrdinalClassifier(cfg.model_id, num_classes=NUM_CLASSES,
                                      mode=mode, dtype=dtype)
    sd_path = os.path.join(ckpt, "model.safetensors")
    sd = (load_file(sd_path) if os.path.exists(sd_path)
          else torch.load(os.path.join(ckpt, "pytorch_model.bin"), map_location="cpu"))
    missing, unexpected = model.load_state_dict(sd, strict=False)
    print(f"missing {len(missing)} | unexpected {len(unexpected)}")
    if missing:
        print("  e.g.", missing[:5])
    assert not any("head" in k for k in missing), "head weights failed to load"

    args = TrainingArguments(output_dir="tmp_eval",
                             per_device_eval_batch_size=cfg.batch_size,
                             bf16=cfg.bf16, report_to=[], do_train=False)
    trainer = OrdinalTrainer(model=model, args=args, mode=mode,
                             processing_class=tokenizer,
                             data_collator=DataCollatorWithPadding(tokenizer),
                             compute_metrics=compute_metrics)

    out = trainer.predict(eval_ds, metric_key_prefix="eval")
    logits = out.predictions
    if isinstance(logits, tuple):
        logits = logits[0]
    logits = torch.as_tensor(np.asarray(logits, dtype=np.float32))
    y_pred = (corn_label_from_logits(logits).numpy() if mode == "corn"
              else logits.argmax(-1).numpy())
    y_true = out.label_ids
    probs = (corn_probas(logits) if mode == "corn"
             else torch.softmax(logits, dim=-1)).numpy()

    bak = f"{output_dir}/dev_metrics.json.bak"
    if os.path.exists(bak):
        old = json.load(open(bak))
        print(f"QWK  old {old.get('eval_qwk')}  new {out.metrics.get('eval_qwk')}")
    print(f"logits {tuple(logits.shape)}  rows sum to 1: "
          f"{np.allclose(probs.sum(1), 1.0, atol=1e-4)}")

    with open(os.path.join(output_dir, "dev_metrics.json"), "w") as f:
        json.dump(out.metrics, f, indent=2)
    with open(os.path.join(output_dir, "dev_predictions.csv"), "w", newline="") as f:
        w = csv.writer(f)
        w.writerow(["true_idx", "pred_idx", "true_label", "pred_label", "p_pred"])
        for i, (t, p) in enumerate(zip(y_true, y_pred)):
            w.writerow([int(t), int(p), LEVELS[int(t)], LEVELS[int(p)],
                        float(probs[i, int(p)])])

    del trainer, model
    gc.collect(); torch.cuda.empty_cache()
    print("saved", output_dir)


run_condition("corn", "runs/t5gemma2_corn")

=== corn | runs/t5gemma2_corn\checkpoint-1900 ===


Loading weights:   0%|          | 0/1327 [00:00<?, ?it/s]

missing 0 | unexpected 0


QWK  old 0.8907117216809263  new 0.8907117216809263
logits (599, 7)  rows sum to 1: True
saved runs/t5gemma2_corn


# Nominal

In [6]:
run_condition("nominal", "runs/t5gemma2_nominal")

=== nominal | runs/t5gemma2_nominal\checkpoint-1900 ===


Loading weights:   0%|          | 0/1327 [00:00<?, ?it/s]

missing 0 | unexpected 0


QWK  old 0.8784819521527945  new 0.8784819521527945
logits (599, 8)  rows sum to 1: True
saved runs/t5gemma2_nominal


# Analysis

In [7]:
from sklearn.metrics import roc_auc_score


def load_preds(run_dir):
    d = pd.read_csv(f"{run_dir}/dev_predictions.csv")
    d = d.rename(columns={"true_idx": "gold", "pred_idx": "pred",
                          "true_label": "gold_label", "pred_label": "pred_label"})
    d["correct"] = (d["gold"] == d["pred"]).astype(int)
    d["adjacent"] = (d["gold"] - d["pred"]).abs().le(1).astype(int)
    return d


def ece(d, n_bins=10):
    b = pd.cut(d["p_pred"], np.linspace(0, 1, n_bins + 1))
    g = d.groupby(b, observed=True)
    return float(((g["correct"].mean() - g["p_pred"].mean()).abs() * g.size()).sum() / len(d))


conds = [("T5Gemma CORN", "runs/t5gemma2_corn"),
         ("T5Gemma Nominal", "runs/t5gemma2_nominal"),
         ("ModernBERT CORN", "runs/modernbert_corn"),
         ("ModernBERT Nominal", "runs/modernbert_nominal")]

rows = []
for name, run_dir in conds:
    if not os.path.exists(f"{run_dir}/dev_predictions.csv"):
        continue
    d = load_preds(run_dir)
    if "p_pred" not in d.columns:
        print(f"{name}: no p_pred")
        continue
    rows.append(dict(condition=name, n=len(d),
                     exact=d["correct"].mean(), adjacent=d["adjacent"].mean(),
                     mean_conf=d["p_pred"].mean(), max_conf=d["p_pred"].max(),
                     conf_correct=d.loc[d["correct"] == 1, "p_pred"].mean(),
                     conf_wrong=d.loc[d["correct"] == 0, "p_pred"].mean(),
                     auroc_exact=roc_auc_score(d["correct"], d["p_pred"]),
                     auroc_adj=roc_auc_score(d["adjacent"], d["p_pred"]),
                     ece=ece(d)))
print(pd.DataFrame(rows).round(3).to_string(index=False))

for name, run_dir in conds:
    if not os.path.exists(f"{run_dir}/dev_predictions.csv"):
        continue
    d = load_preds(run_dir)
    if "p_pred" not in d.columns:
        continue
    b = pd.cut(d["p_pred"], [0, .3, .4, .5, .6, .7, 1.0])
    print(f"\n{name}")
    print(d.groupby(b, observed=True).agg(
        n=("correct", "size"), exact=("correct", "mean"),
        adjacent=("adjacent", "mean"), mean_conf=("p_pred", "mean")).round(3))

         condition   n  exact  adjacent  mean_conf  max_conf  conf_correct  conf_wrong  auroc_exact  auroc_adj   ece
      T5Gemma CORN 599  0.556     0.960      0.925     1.000         0.929       0.919        0.539      0.586 0.369
   T5Gemma Nominal 599  0.543     0.950      0.923     1.000         0.937       0.907        0.585      0.626 0.381
   ModernBERT CORN 599  0.487     0.930      0.470     0.673         0.487       0.454        0.614      0.728 0.054
ModernBERT Nominal 599  0.516     0.933      0.485     0.709         0.496       0.473        0.588      0.687 0.038

T5Gemma CORN
              n  exact  adjacent  mean_conf
p_pred                                     
(0.5, 0.6]   24  0.458     0.917      0.554
(0.6, 0.7]   29  0.621     1.000      0.649
(0.7, 1.0]  546  0.557     0.960      0.956

T5Gemma Nominal
              n  exact  adjacent  mean_conf
p_pred                                     
(0.4, 0.5]    5  0.400     1.000      0.500
(0.5, 0.6]   26  0.423     1.000

# T5Gemma Corn's overconfidence

In [8]:
d = load_preds("runs/t5gemma2_corn")
print(d.groupby("gold_label")["p_pred"].agg(["size", "mean"]).round(3).reindex(LEVELS))
print(d[d["p_pred"] > 0.99]["correct"].agg(["size", "mean"]).round(3))

            size   mean
gold_label             
A2            30  0.942
A2+          136  0.933
B1           128  0.910
B1+           95  0.928
B2           110  0.932
B2+           49  0.908
C1            37  0.924
C1+           14  0.926
size    304.000
mean      0.582
Name: correct, dtype: float64


# Results for poster

In [9]:
from metrics import evaluate_predictions

METRICS = ["qwk", "mae", "exact_acc", "adjacent_acc", "f1_macro"]

cols = {}
for name, run_dir in conds:
    d = load_preds(run_dir)
    m = evaluate_predictions(d["gold"].to_numpy(), d["pred"].to_numpy())
    m = {("exact_acc" if k == "accuracy" else
          "adjacent_acc" if k == "adjacent_accuracy" else k): v
         for k, v in m.items()}
    m["ece"] = ece(d)
    m["mean_conf"] = d["p_pred"].mean()
    cols[name] = m

table = pd.DataFrame(cols).loc[METRICS + ["ece", "mean_conf"]].round(3)
table.to_csv("runs/poster_encoder_table.csv")
table

,T5Gemma CORN,T5Gemma Nominal,ModernBERT CORN,ModernBERT Nominal
qwk,0.891,0.878,0.849,0.858
mae,0.484,0.509,0.596,0.564
exact_acc,0.556,0.543,0.487,0.516
adjacent_acc,0.960,0.950,0.930,0.933
f1_macro,0.447,0.399,0.339,0.364
ece,0.369,0.381,0.054,0.038
mean_conf,0.925,0.923,0.470,0.485
